# 02 - Dataset Construction

## Objective

The objective of this notebook is to transform the raw Polymarket data collected in `01_Data_Acquisition.ipynb` into a clean dataset.

The raw data consists of two main components:

1. Historical implied probability time series.
2. Market-level metadata.

The goal is to combine both sources into a structured dataset suitable for exploratory analysis, calibration studies, Bayesian probability estimation, and future signal generation.

---

## Inputs

This notebook uses the following datasets generated in the previous phase:

```text
data/raw/prices_raw.csv
data/raw/markets_metadata.csv
```

--- 

### prices_raw.csv

Contains historical implied probabilities for each market.

fields

| Column                | Description                                          |
| --------------------- | ---------------------------------------------------- |
| `timestamp`           | Timestamp of the observed market-implied probability |
| `implied_probability` | YES contract price interpreted as probability        |
| `market_id`           | Unique Polymarket market identifier                  |
| `question`            | Market question                                      |
| `slug`                | Market slug                                          |

 

### markets_metadata.csv

Contains market-level information.

fields:

| Column          | Description                         |
| --------------- | ----------------------------------- |
| `id`            | Unique Polymarket market identifier |
| `question`      | Market question                     |
| `endDate`       | Market resolution or closing date   |
| `volume`        | Total traded volume                 |
| `liquidity`     | Market liquidity                    |
| `outcomes`      | Possible market outcomes            |
| `outcomePrices` | Final outcome prices                |
| `yes_token_id`  | YES outcome CLOB token identifier   |


--- 


## Dataset Construction Logic

The construction process follows four steps:

1. Load raw historical probabilities and market metadata.
2. Validate data types, timestamps, and identifiers.
3. Aggregate historical probability series at the market level.
4. Merge market-level probability features with metadata and final outcomes.


The final output of this notebook will be a market-level dataset where each row represents one resolved prediction market.

structure:

| market_id | question | start_date | end_date | final_probability | mean_probability | probability_volatility | volume | liquidity | outcome |
| --------- | -------- | ---------- | -------- | ----------------- | ---------------- | ---------------------- | ------ | --------- | ------- |


---

This dataset will be used in the next notebooks for:

1. Exploratory analysis
2. Calibration analysis
3. Brier Score evaluation
4. Bayesian fair probability estimation
5. Mispricing detection
6. Backtesting

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
# File paths
PRICES_PATH = "../data/raw/prices_raw.csv"
METADATA_PATH = "../data/raw/markets_metadata.csv"
# Load raw datasets
df_prices = pd.read_csv(PRICES_PATH)
df_metadata = pd.read_csv(METADATA_PATH)
display(df_prices.head(2))
display(df_metadata.head(2))

,timestamp,implied_probability,market_id,question,slug
0,2023-02-07 01:00:32+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...
1,2023-02-07 02:00:53+00:00,0.5,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...


,id,question,slug,category,endDate,closed,volume,liquidity,outcomes,outcomePrices,clobTokenIds,yes_token_id,endDate_dt
0,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,NaN,2023-05-01T00:00:00Z,True,35102.732603,25.00501,"[""Yes"", ""No""]","[""0"", ""1""]","[""38374278491791697742783566758802364186445803...",3837427849179169774278356675880236418644580318...,2023-05-01 00:00:00+00:00
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,NaN,2023-04-02T00:00:00Z,True,500.000000,0.00000,"[""Reds"", ""Pirates""]","[""1"", ""0""]","[""80040235635822024858376428020052595519279817...",8004023563582202485837642802005259551927981755...,2023-04-02 00:00:00+00:00


## Basic Data Validation

In [2]:
print(df_prices.info())

print("\n")
print(df_metadata.info())

print("\n")
display(df_prices.isna().sum())

print("\n")
display(df_metadata.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 100642 entries, 0 to 100641
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   timestamp            100642 non-null  str    
 1   implied_probability  100642 non-null  float64
 2   market_id            100642 non-null  int64  
 3   question             100642 non-null  str    
 4   slug                 100642 non-null  str    
dtypes: float64(1), int64(1), str(3)
memory usage: 3.8 MB
None


<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             43 non-null     int64  
 1   question       43 non-null     str    
 2   slug           43 non-null     str    
 3   category       0 non-null      float64
 4   endDate        43 non-null     str    
 5   closed         43 non-null     bool   
 6   volume         43 non-null    

timestamp              0
implied_probability    0
market_id              0
question               0
slug                   0
dtype: int64

id                0
question          0
slug              0
category         43
endDate           0
closed            0
volume            0
liquidity        18
outcomes          0
outcomePrices     0
clobTokenIds      0
yes_token_id      0
endDate_dt        0
dtype: int64

In [3]:
print("unique markets in prices:")
print(df_prices["market_id"].nunique())
print("\nrows in metadata:")
print(len(df_metadata))
print("\nduplicate market:")
print(df_metadata["id"].duplicated().sum())

unique markets in prices:
43

rows in metadata:
43

duplicate market:
0


In [4]:
df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"],utc=True)
print("min timestamp:",df_prices["timestamp"].min())
print("max timestamp:",df_prices["timestamp"].max())

min timestamp: 2023-02-07 01:00:32+00:00
max timestamp: 2024-12-30 23:00:04+00:00


# type cleaning and probability validation 


In [ ]:
# Convert timestamps and dates

df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"],utc=True,errors="coerce")

df_metadata["endDate_dt"] = pd.to_datetime(df_metadata["endDate_dt"],utc=True,errors="coerce")

# Ensure numeric fields

df_prices["implied_probability"] = pd.to_numeric(df_prices["implied_probability"],errors="coerce")

df_metadata["volume"] = pd.to_numeric(df_metadata["volume"],errors="coerce")

df_metadata["liquidity"] = pd.to_numeric(df_metadata["liquidity"],errors="coerce")

# Validate probability bounds
invalid_probs = df_prices[(df_prices["implied_probability"] < 0) | (df_prices["implied_probability"] > 1)]
print("Invalid probability rows:", len(invalid_probs))


Invalid probability rows: 0


# aggregation by market 

1. first_probability: first observed probability.
2. final_probability: last obsererved probability before close.
3. mean_probability:  average probability during observed life.
4. probability_volatility:  how much did the probability  change? 
5. n_observations: data quality and preparation.

In [7]:

df_market_features = (df_prices.groupby("market_id").agg(start_date=("timestamp", "min"),
                                                         last_observed_date=("timestamp", "max"),
                                                         n_observations=("implied_probability", "count"),
                                                         first_probability=("implied_probability", "first"),
                                                         final_probability=("implied_probability", "last"),
                                                         mean_probability=("implied_probability", "mean"),
                                                         median_probability=("implied_probability", "median"),
                                                         min_probability=("implied_probability", "min"),
                                                         max_probability=("implied_probability", "max"),
                                                         probability_volatility=("implied_probability", "std")).reset_index())

print("market-level features shape:", df_market_features.shape)

display(df_market_features.head(3))

market-level features shape: (43, 11)


,market_id,start_date,last_observed_date,n_observations,first_probability,final_probability,mean_probability,median_probability,min_probability,max_probability,probability_volatility
0,248594,2023-02-07 01:00:32+00:00,2023-04-30 23:00:11+00:00,1826,0.5,0.02,0.136156,0.09,0.015,0.93,0.159504
1,249778,2023-03-31 19:00:37+00:00,2023-04-01 23:00:34+00:00,23,0.5,0.55,0.514348,0.50,0.500,0.78,0.058839
2,250474,2023-04-24 23:00:21+00:00,2023-04-25 23:00:16+00:00,23,0.4,0.36,0.355652,0.36,0.315,0.40,0.029128


The df_market_features is already becoming a readable market time series table.

In [ ]:
# Market-Level Feature Table
df_master = df_market_features.merge(df_metadata,left_on="market_id",right_on="id",how="left",validate="one_to_one"
)
print(df_master.shape)
display(df_master.head(3))

(43, 24)


,market_id,start_date,last_observed_date,n_observations,first_probability,final_probability,mean_probability,median_probability,min_probability,max_probability,probability_volatility,id,question,slug,category,endDate,closed,volume,liquidity,outcomes,outcomePrices,clobTokenIds,yes_token_id,endDate_dt
0,248594,2023-02-07 01:00:32+00:00,2023-04-30 23:00:11+00:00,1826,0.5,0.02,0.136156,0.09,0.015,0.93,0.159504,248594,Will Hunter Biden be federally indicted by May...,will-hunter-biden-be-federally-indicted-by-may...,NaN,2023-05-01T00:00:00Z,True,35102.732603,25.00501,"[""Yes"", ""No""]","[""0"", ""1""]","[""38374278491791697742783566758802364186445803...",3837427849179169774278356675880236418644580318...,2023-05-01 00:00:00+00:00
1,249778,2023-03-31 19:00:37+00:00,2023-04-01 23:00:34+00:00,23,0.5,0.55,0.514348,0.50,0.500,0.78,0.058839,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,mlb-cin-pit-2023-04-02,NaN,2023-04-02T00:00:00Z,True,500.000000,0.00000,"[""Reds"", ""Pirates""]","[""1"", ""0""]","[""80040235635822024858376428020052595519279817...",8004023563582202485837642802005259551927981755...,2023-04-02 00:00:00+00:00
2,250474,2023-04-24 23:00:21+00:00,2023-04-25 23:00:16+00:00,23,0.4,0.36,0.355652,0.36,0.315,0.40,0.029128,250474,Did US GDP grow 2.5% or more in Q1 2023?,did-us-gdp-grow-2pt5-or-more-in-q1-2023,NaN,2023-04-26T00:00:00Z,True,3817.688096,0.00000,"[""Yes"", ""No""]","[""0"", ""1""]","[""26005592608097290511618522220205731503687994...",2600559260809729051161852222020573150368799420...,2023-04-26 00:00:00+00:00


# ready outcome  : From time series to supervised problem 

In [9]:
import ast


def extract_binary_outcome(price_string):
    """
    Convert final outcomePrices into binary outcome.

    Returns:
        1 -> YES occurred
        0 -> NO occurred
        NaN -> unknown
    """

    try:

        prices = ast.literal_eval(price_string)

        if len(prices) < 2:
            return np.nan

        yes_price = float(prices[0])

        if yes_price >= 0.99:
            return 1

        elif yes_price <= 0.01:
            return 0

        return np.nan

    except Exception:
        return np.nan

In [10]:
df_master["outcome"] = (df_master["outcomePrices"].apply(extract_binary_outcome))
print(df_master["outcome"].value_counts(dropna=False))

display(df_master[[
            "market_id",
            "question",
            "final_probability",
            "outcomePrices",
            "outcome"]].head(4))

outcome
0    29
1    14
Name: count, dtype: int64


,market_id,question,final_probability,outcomePrices,outcome
0,248594,Will Hunter Biden be federally indicted by May...,0.020,"[""0"", ""1""]",0
1,249778,MLB: Cincinnati Reds vs. Pittsburgh Pirates 20...,0.550,"[""1"", ""0""]",1
2,250474,Did US GDP grow 2.5% or more in Q1 2023?,0.360,"[""0"", ""1""]",0
3,251025,Will Ron DeSantis file to run for president by...,0.995,"[""1"", ""0""]",1


In [ ]:
# First feature enginnering 
df_master["market_duration_days"] = (pd.to_datetime(df_master["last_observed_date"])-pd.to_datetime(df_master["start_date"])).dt.days
df_master["probability_change"] = (df_master["final_probability"]-df_master["first_probability"])
df_master["abs_probability_change"] = (df_master["probability_change"].abs())
df_master["surprise"] = (df_master["outcome"]-df_master["final_probability"])
df_master["abs_surprise"] = (df_master["surprise"].abs())

In [12]:
df_master.to_csv("../data/processed/market_dataset.csv",index=False)